In [18]:
import pandas as pd

# Read rental_school_hospital_data, postcode info data and rental_crime data

In [19]:
rental_school_hospital = pd.read_csv("../data/curated/rental_data_distance_sf.csv")
rental_crime = pd.read_csv("../data/curated/rental_data_crime_join.csv")
postcode_2011 = pd.read_csv("../data/curated/postcode_info_2011.csv")
postcode_2016 = pd.read_csv("../data/curated/postcode_info_2016.csv")
postcode_2021 = pd.read_csv("../data/curated/postcode_info_2021.csv")


In [20]:
# lowercase column names
postcode_2011.columns = postcode_2011.columns.str.lower()
postcode_2016.columns = postcode_2016.columns.str.lower()
postcode_2021.columns = postcode_2021.columns.str.lower()
rental_crime.columns = rental_crime.columns.str.lower()
rental_school_hospital.columns = rental_school_hospital.columns.str.lower()

In [21]:
# drop duplicate properties
rental_school_hospital = rental_school_hospital.drop_duplicates(
    subset=["name", "cost_text", "coordinates", "property type", "beds", "baths", "parking", 
            ])

rental_crime = rental_crime.drop_duplicates(
    subset=["name", "cost_text", "coordinates", "property type", "beds", "baths", "parking"])

In [22]:
# lowercase column names
postcode_2011.columns = postcode_2011.columns.str.lower()
postcode_2016.columns = postcode_2016.columns.str.lower()
postcode_2021.columns = postcode_2021.columns.str.lower()


# add year in column name for postcode info dataframes
postcode_2011.columns = [col + " 2011" for col in postcode_2011.columns]
postcode_2016.columns = [col + " 2016" for col in postcode_2016.columns]
postcode_2021.columns = [col + " 2021" for col in postcode_2021.columns]

In [23]:
# impute missing values as 0
rental_crime["offence count 2023"] = rental_crime["offence count 2023"].fillna(0)
rental_crime["offence count 2022"] = rental_crime["offence count 2022"].fillna(0)
rental_crime["offence count 2021"] = rental_crime["offence count 2021"].fillna(0)

# Merge rental_data_school_hopsital with rental_crime

In [24]:
print(len(rental_school_hospital))
# perform merge
rental_merged = pd.merge(rental_school_hospital, rental_crime, on=["name", "property type", "coordinates", "weekly rent amount",
                                                                   "beds", "baths", "parking", "title", "postcode", 
                                                                   "property features", "description", "nbn"], how="inner", 
                         suffixes=('_left', '_right'))

# drop columns that are duplicates
for col_name in rental_merged.columns:
    col_name_list = col_name.split("_")
    if col_name_list[-1] == "right":
        rental_merged = rental_merged.drop(col_name, axis=1)
    # rename column name to original
    if col_name_list[-1] == "left":
        rental_merged.rename(columns={col_name: "_".join(col_name_list[0:-1])}, inplace=True)
        
print(len(rental_merged))
rental_merged.columns

11335
11337


Index(['name', 'cost_text', 'coordinates', 'property type', 'beds', 'baths',
       'parking', 'title', 'description', 'property features', 'nbn',
       'weekly rent amount', 'postcode', 'geometry', 'sa2_code21',
       'sa2_name21', 'closest primary school', 'distance primary school',
       'closest secondary school', 'distance secondary school',
       'closest university campus', 'distance university campus',
       'closest hospital', 'distance to closest hospital',
       'offence count 2023', 'offence count 2022', 'offence count 2021'],
      dtype='object')

# Merge different years' postcode data together 

In [25]:
# merge 2011 and 2016
postcode_merged = pd.merge(postcode_2011, postcode_2016, left_on = "postcode 2011", right_on = "postcode 2016",
                           how="inner")
postcode_merged = postcode_merged.drop("postcode 2011", axis=1)

postcode_merged

,estimated resident population (no.) 2011,employed (no.) 2011,unemployed (no.) 2011,in the labour force (no.) 2011,total persons aged 15 years and over (no.) 2011,land area (ha) 2011,total families (no.) 2011,total dwellings (no.) 2011,median equivalised total household income (weekly) ($) 2011,median weekly household rental payment ($) 2011,...,employed (no.) 2016,unemployed (no.) 2016,in the labour force (no.) 2016,total persons aged 15 years and over (no.) 2016,land area (ha) 2016,total families (no.) 2016,total dwellings (no.) 2016,median equivalised total household income (weekly) ($) 2016,median weekly household rental payment ($) 2016,average family size (no. of persons) 2016
0,0.0,55833.0,4270.0,60108.0,97390.0,0.0,23110.0,0.0,1017.090909,402.090909,...,67942.0,7596.0,75528.0,131669.0,0.0,28516.0,0.0,1045.181818,441.636364,2.500
1,0.0,3122.0,109.0,3234.0,4630.0,0.0,1087.0,0.0,1504.000000,420.000000,...,3030.0,123.0,3151.0,4670.0,0.0,1102.0,0.0,1759.000000,462.000000,2.400
2,0.0,7840.0,706.0,8558.0,13921.0,0.0,3015.0,0.0,1014.000000,395.000000,...,9948.0,1157.0,11107.0,18679.0,0.0,3803.0,0.0,1031.500000,418.500000,2.550
3,0.0,38469.0,2536.0,41011.0,63808.0,0.0,15049.0,0.0,1120.000000,417.625000,...,48505.0,4901.0,53399.0,89682.0,0.0,19150.0,0.0,1152.750000,456.125000,2.400
4,0.0,12197.0,696.0,12899.0,19496.0,0.0,4936.0,0.0,1257.000000,483.500000,...,16831.0,1295.0,18128.0,28030.0,0.0,6728.0,0.0,1371.250000,501.000000,2.375
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,0.0,4020.0,172.0,4192.0,6995.0,0.0,2434.0,0.0,630.000000,195.000000,...,4009.0,225.0,4235.0,7397.0,0.0,2454.0,0.0,681.000000,230.000000,2.900
423,0.0,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,...,8397.0,597.0,8994.0,18640.0,0.0,5794.0,0.0,616.000000,244.000000,2.800
424,0.0,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,...,8397.0,597.0,8994.0,18640.0,0.0,5794.0,0.0,616.000000,244.000000,2.800
425,0.0,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,...,8397.0,597.0,8994.0,18640.0,0.0,5794.0,0.0,616.000000,244.000000,2.800


In [26]:
# merge 2021 data with rest of postcode data
postcode_merged = pd.merge(postcode_merged, postcode_2021, left_on = "postcode 2016", right_on = "postcode 2021",
                           how="inner")
postcode_merged = postcode_merged.drop("postcode 2016", axis=1)
postcode_merged

,estimated resident population (no.) 2011,employed (no.) 2011,unemployed (no.) 2011,in the labour force (no.) 2011,total persons aged 15 years and over (no.) 2011,land area (ha) 2011,total families (no.) 2011,total dwellings (no.) 2011,median equivalised total household income (weekly) ($) 2011,median weekly household rental payment ($) 2011,...,employed (no.) 2021,unemployed (no.) 2021,in the labour force (no.) 2021,total persons aged 15 years and over (no.) 2021,land area (ha) 2021,total families (no.) 2021,total dwellings (no.) 2021,median equivalised total household income (weekly) ($) 2021,median weekly household rental payment ($) 2021,average family size (no. of persons) 2021
0,0.0,55833.0,4270.0,60108.0,97390.0,0.0,23110.0,0.0,1017.090909,402.090909,...,88622.0,6958.0,95571.0,140650.0,2408.8,33336.0,99364.0,1343.818182,412.727273,2.50
1,0.0,3122.0,109.0,3234.0,4630.0,0.0,1087.0,0.0,1504.000000,420.000000,...,3078.0,100.0,3178.0,4609.0,290.0,1199.0,3453.0,1921.000000,480.000000,2.30
2,0.0,7840.0,706.0,8558.0,13921.0,0.0,3015.0,0.0,1014.000000,395.000000,...,13642.0,994.0,14639.0,20764.0,325.0,4978.0,14049.0,1309.000000,385.500000,2.50
3,0.0,38469.0,2536.0,41011.0,63808.0,0.0,15049.0,0.0,1120.000000,417.625000,...,64480.0,4738.0,69220.0,98904.0,1360.5,22651.0,74682.0,1403.250000,410.375000,2.40
4,0.0,12197.0,696.0,12899.0,19496.0,0.0,4936.0,0.0,1257.000000,483.500000,...,23396.0,1365.0,24773.0,33303.0,1118.7,8593.0,25941.0,1602.750000,461.000000,2.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,0.0,4020.0,172.0,4192.0,6995.0,0.0,2434.0,0.0,630.000000,195.000000,...,4596.0,166.0,4769.0,8128.0,60069.1,2721.0,4366.0,871.000000,280.000000,2.90
423,0.0,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,...,10846.0,451.0,11301.0,22747.0,72876.3,7364.0,16030.0,767.000000,298.000000,2.70
424,0.0,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,...,10846.0,451.0,11301.0,22747.0,72876.3,7364.0,16030.0,767.000000,298.000000,2.70
425,0.0,8111.0,391.0,8503.0,16628.0,0.0,5314.0,0.0,582.000000,230.000000,...,10846.0,451.0,11301.0,22747.0,72876.3,7364.0,16030.0,767.000000,298.000000,2.70


# Merge postcode data with rental data

In [27]:
print(len(rental_merged))
rental_merged = pd.merge(rental_merged, postcode_merged, left_on = "postcode", right_on = "postcode 2021", 
                              how="left")
rental_merged = rental_merged.drop("postcode 2021", axis=1)
print(len(rental_merged))
rental_merged

11337
11337


,name,cost_text,coordinates,property type,beds,baths,parking,title,description,property features,...,employed (no.) 2021,unemployed (no.) 2021,in the labour force (no.) 2021,total persons aged 15 years and over (no.) 2021,land area (ha) 2021,total families (no.) 2021,total dwellings (no.) 2021,median equivalised total household income (weekly) ($) 2021,median weekly household rental payment ($) 2021,average family size (no. of persons) 2021
0,Rooms/227 Elgar Road Surrey Hills VIC 3127,$120.00,"[-37.8353315, 145.1114189]",House,1,2,6,Single room in rooming house - CURRENTLY NO RO...,"['** BOOK INSPECTION ONLINE **', '*Currently n...","['Furnished*', 'Broadband internet access*', '...",...,20755.0,960.0,21716.0,35555.0,1461.6,11741.0,17800.0,1495.00,466.00,3.033333
1,30 Daly Boulevard Highton VIC 3216,$130,"[-38.1962932, 144.3060777]",House,9,2,2,Student Accommodation - 1 Room Available!,['We have 1 Rooms Available to Rent.Room 2 - $...,"['Gas*', 'Internal Laundry*', 'Secure Parking*...",...,38130.0,1594.0,39722.0,60483.0,10566.0,19375.0,29407.0,1116.25,381.25,3.000000
2,Berth 5274 Martha Cove Waterway Safety Beach V...,$140 per week,"[-38.3138012, 145.0045603]",House,1,1,0,Boat Berth B5274,['BOAT BERTH AVAILABLE-B5274-Located in Martha...,[],...,5832.0,206.0,6042.0,11569.0,4505.8,3798.0,8829.0,931.00,405.00,2.700000
3,18 Cedarville Close Highton VIC 3216,$140,"[-38.19508, 144.3154857]",House,1,2,6,Room 4 rent!,['Rooms for rent in this student share house. ...,"['Built in wardrobes', 'Internal Laundry', 'Ga...",...,38130.0,1594.0,39722.0,60483.0,10566.0,19375.0,29407.0,1116.25,381.25,3.000000
4,123 Thornhill Road Highton VIC 3216,$150,"[-38.1845196, 144.3107906]",House,1,4,3,Room for rent near Uni,['Rooms within walking distance to Waurn Ponds...,"['Ensuite', 'Furnished', 'Built in wardrobes*'...",...,38130.0,1594.0,39722.0,60483.0,10566.0,19375.0,29407.0,1116.25,381.25,3.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11332,Enquire For Address Brighton VIC 3186,"$4,400.00","[-37.9044002, 144.9997396]",House,4,4,2,Golden Mile Luxury,"['Inspect by Appointment Only ', 'This family ...","['Study', 'Swimming Pool', 'Ducted Cooling', '...",...,20251.0,794.0,21054.0,34210.0,1388.5,11199.0,18176.0,1678.00,535.50,3.000000
11333,335 The Esplanade Indented Head VIC 3223,"$4,700 pw /$700 pn High, $3,775 pw/$545 pn Low","[-38.1439923, 144.7147899]",House,4,3,0,"335 The Esplanade, Indented Head",['This stylish 4 bedroom home will surely make...,"['Ensuite', 'Deck', 'Dishwasher', 'Ducted Heat...",...,3485.0,169.0,3655.0,8532.0,7185.6,2799.0,7449.0,773.00,340.00,2.500000
11334,8701/7 Riverside Quay Southbank VIC 3006,"$5,000.00","[-37.8213342, 144.9646982]",Apartment / Unit / Flat,3,3,4,On Top of the World,"[""A brilliant opportunity to lease the exclusi...",[],...,23396.0,1365.0,24773.0,33303.0,1118.7,8593.0,25941.0,1602.75,461.00,2.450000
11335,1 Weston Street Balwyn VIC 3103,"$9,950 weekly","[-37.8117577, 145.0775507]",House,6,7,8,"Lifestyle, Modern & Luxury Living Awaits",['**Scheduling your property inspection is now...,"['Air conditioning*', 'Built in wardrobes*', '...",...,17455.0,913.0,18372.0,30952.0,1443.4,10395.0,14928.0,1294.00,507.50,3.100000


In [28]:
rental_merged.isna().any()

name                                                           False
cost_text                                                      False
coordinates                                                    False
property type                                                  False
beds                                                           False
baths                                                          False
parking                                                        False
title                                                          False
description                                                    False
property features                                              False
nbn                                                            False
weekly rent amount                                             False
postcode                                                       False
geometry                                                       False
sa2_code21                        

# Merge town data with rental data

In [29]:
town_data = pd.read_csv("../data/curated/rental_data_town_join.csv")
# lowercase column names
town_data.columns = town_data.columns.str.lower()
print(town_data.columns)
# drop duplicate rows
print(len(town_data))
town_data = town_data.drop_duplicates(
    subset=["name", "cost_text", "coordinates", "property type", "beds", "baths", "parking"])
print(len(town_data))
town_data

Index(['unnamed: 0', 'name', 'cost_text', 'coordinates', 'property type',
       'beds', 'baths', 'parking', 'title', 'description', 'property features',
       'nbn', 'weekly rent amount', 'postcode', 'geometry', 'sa2_code21',
       'sa2_name21', 'total_supermarkets', 'major_supermarkets', 'city_centre',
       'closest_city_centre', 'coastal', 'closest_cc_dist', 'dist_to_melb'],
      dtype='object')
11409
11335


,unnamed: 0,name,cost_text,coordinates,property type,beds,baths,parking,title,description,...,geometry,sa2_code21,sa2_name21,total_supermarkets,major_supermarkets,city_centre,closest_city_centre,coastal,closest_cc_dist,dist_to_melb
0,0,Rooms/227 Elgar Road Surrey Hills VIC 3127,$120.00,"[-37.8353315, 145.1114189]",House,1,2,6,Single room in rooming house - CURRENTLY NO RO...,"['** BOOK INSPECTION ONLINE **', '*Currently n...",...,POINT (145.1114189 -37.8353315),207031167.0,Surrey Hills (East) - Mont Albert,2.0,0.0,False,Melbourne,False,11.566742,12.618789
1,1,30 Daly Boulevard Highton VIC 3216,$130,"[-38.1962932, 144.3060777]",House,9,2,2,Student Accommodation - 1 Room Available!,['We have 1 Rooms Available to Rent.Room 2 - $...,...,POINT (144.3060777 -38.1962932),203021042.0,Highton,14.0,0.0,False,Geelong,False,7.459818,71.172437
2,2,Berth 5274 Martha Cove Waterway Safety Beach V...,$140 per week,"[-38.3138012, 145.0045603]",House,1,1,0,Boat Berth B5274,['BOAT BERTH AVAILABLE-B5274-Located in Martha...,...,POINT (145.0045603 -38.3138012),214021377.0,Dromana,1.0,0.0,False,Cranbourne,False,37.174742,58.248165
3,3,18 Cedarville Close Highton VIC 3216,$140,"[-38.19508, 144.3154857]",House,1,2,6,Room 4 rent!,['Rooms for rent in this student share house. ...,...,POINT (144.3154857 -38.19508),203021042.0,Highton,14.0,0.0,False,Geelong,False,7.459818,71.172437
4,4,123 Thornhill Road Highton VIC 3216,$150,"[-38.1845196, 144.3107906]",House,1,4,3,Room for rent near Uni,['Rooms within walking distance to Waurn Ponds...,...,POINT (144.3107906 -38.1845196),203021042.0,Highton,14.0,0.0,False,Geelong,False,7.459818,71.172437
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11404,11404,Enquire For Address Brighton VIC 3186,"$4,400.00","[-37.9044002, 144.9997396]",House,4,4,2,Golden Mile Luxury,"['Inspect by Appointment Only ', 'This family ...",...,POINT (144.9997396 -37.9044002),208011169.0,Brighton (Vic.),8.0,0.0,False,Melbourne,False,7.399952,10.876836
11405,11405,335 The Esplanade Indented Head VIC 3223,"$4,700 pw /$700 pn High, $3,775 pw/$545 pn Low","[-38.1439923, 144.7147899]",House,4,3,0,"335 The Esplanade, Indented Head",['This stylish 4 bedroom home will surely make...,...,POINT (144.7147899 -38.1439923),203031051.0,Portarlington,2.0,0.0,False,Geelong,True,27.298679,44.708244
11406,11406,8701/7 Riverside Quay Southbank VIC 3006,"$5,000.00","[-37.8213342, 144.9646982]",Apartment / Unit / Flat,3,3,4,On Top of the World,"[""A brilliant opportunity to lease the exclusi...",...,POINT (144.9646982 -37.8213342),206041508.0,Southbank (West) - South Wharf,1.0,0.0,False,Melbourne,False,1.775904,1.827457
11407,11407,1 Weston Street Balwyn VIC 3103,"$9,950 weekly","[-37.8117577, 145.0775507]",House,6,7,8,"Lifestyle, Modern & Luxury Living Awaits",['**Scheduling your property inspection is now...,...,POINT (145.0775507 -37.8117577),207011147.0,Balwyn,4.0,0.0,False,Melbourne,False,9.486256,10.159737


In [30]:
rental_merged = pd.merge(rental_merged, town_data, on=["name", "property type", "coordinates", "weekly rent amount",
                                                                   "beds", "baths", "parking", "title"], how="inner", 
                         suffixes=('_left', '_right'))

# drop columns that are duplicates
for col_name in rental_merged.columns:
    col_name_list = col_name.split("_")
    if col_name_list[-1] == "right":
        rental_merged = rental_merged.drop(col_name, axis=1)
    # rename column name to original
    if col_name_list[-1] == "left":
        rental_merged.rename(columns={col_name: "_".join(col_name_list[0:-1])}, inplace=True)

In [31]:
rental_merged = rental_merged.drop(["title", "description", "cost_text", "unnamed: 0"], axis=1)
print(list(rental_merged.columns))

['name', 'coordinates', 'property type', 'beds', 'baths', 'parking', 'property features', 'nbn', 'weekly rent amount', 'postcode', 'geometry', 'sa2_code21', 'sa2_name21', 'closest primary school', 'distance primary school', 'closest secondary school', 'distance secondary school', 'closest university campus', 'distance university campus', 'closest hospital', 'distance to closest hospital', 'offence count 2023', 'offence count 2022', 'offence count 2021', 'estimated resident population (no.) 2011', 'employed (no.) 2011', 'unemployed (no.) 2011', 'in the labour force (no.) 2011', 'total persons aged 15 years and over (no.) 2011', 'land area (ha) 2011', 'total families (no.) 2011', 'total dwellings (no.) 2011', 'median equivalised total household income (weekly) ($) 2011', 'median weekly household rental payment ($) 2011', 'average family size (no. of persons) 2011', 'estimated resident population (no.) 2016', 'employed (no.) 2016', 'unemployed (no.) 2016', 'in the labour force (no.) 2016'

In [32]:
# remove columns with just 0
dropped_columns_list = []
for col_name in rental_merged.columns:
    if (rental_merged[col_name] == 0).all():
        dropped_columns_list.append(col_name)
        rental_merged = rental_merged.drop(col_name, axis=1)
print(dropped_columns_list)
print(list(rental_merged.columns))

['estimated resident population (no.) 2011', 'land area (ha) 2011', 'total dwellings (no.) 2011', 'estimated resident population (no.) 2016', 'land area (ha) 2016', 'total dwellings (no.) 2016']
['name', 'coordinates', 'property type', 'beds', 'baths', 'parking', 'property features', 'nbn', 'weekly rent amount', 'postcode', 'geometry', 'sa2_code21', 'sa2_name21', 'closest primary school', 'distance primary school', 'closest secondary school', 'distance secondary school', 'closest university campus', 'distance university campus', 'closest hospital', 'distance to closest hospital', 'offence count 2023', 'offence count 2022', 'offence count 2021', 'employed (no.) 2011', 'unemployed (no.) 2011', 'in the labour force (no.) 2011', 'total persons aged 15 years and over (no.) 2011', 'total families (no.) 2011', 'median equivalised total household income (weekly) ($) 2011', 'median weekly household rental payment ($) 2011', 'average family size (no. of persons) 2011', 'employed (no.) 2016', 'un

In [33]:
# create employment ratio column
rental_merged["employment ratio 2011"] = rental_merged["employed (no.) 2011"]/ (rental_merged["employed (no.) 2011"] + rental_merged["unemployed (no.) 2011"])
rental_merged["employment ratio 2016"] = rental_merged["employed (no.) 2016"]/ (rental_merged["employed (no.) 2016"] + rental_merged["unemployed (no.) 2016"])
rental_merged["employment ratio 2021"] = rental_merged["employed (no.) 2021"]/ (rental_merged["employed (no.) 2021"] + rental_merged["unemployed (no.) 2021"])

# drop columns we do not need anymore
rental_merged = rental_merged.drop(["employed (no.) 2011", "unemployed (no.) 2011", "employed (no.) 2016", "unemployed (no.) 2016", 
           "employed (no.) 2021", "unemployed (no.) 2021"], axis=1)

In [34]:
rental_merged.to_csv("../data/curated/merged_data.csv", index=False)

In [36]:
rental_merged[rental_merged.isna().any(axis=1)].to_csv("../data/curated/null_rows_delete.csv")

In [38]:
town = pd.read_csv("../data/curated/rental_data_town_join.csv")

In [39]:
town[town.isna().any(axis=1)]

,Unnamed: 0,name,cost_text,coordinates,property type,beds,baths,parking,title,description,...,geometry,SA2_CODE21,SA2_NAME21,Total_Supermarkets,Major_Supermarkets,city_centre,closest_city_centre,coastal,closest_cc_dist,dist_to_melb
54,54,Unit 2/6 Northill Avenue Strathmerton VIC 3641,$200 per week,"[-35.9277533, 145.4796393]",Apartment / Unit / Flat,1,1,1,One Bedroom Unit,['Neat one bedroom unit in block of only three...,...,POINT (145.4796393 -35.9277533),216021413.0,Numurkah,NaN,NaN,False,Shepparton,False,55.967719,219.493555
61,61,3/9 Camp Street Donald VIC 3480,$210 PER WEEK,"[-36.3700135, 142.9747066]",Apartment / Unit / Flat,2,1,0,"UNIT FOR LEASE - 3/9 CAMP STREET, DONALD",['* Two bedroom unit for lease.* Open plan liv...,...,POINT (142.9747066 -36.3700135),215031400.0,Buloke,NaN,NaN,False,Horsham,False,77.858022,243.561814
62,62,2/9 Camp Street Donald VIC 3480,$210 PER WEEK,"[-36.3700135, 142.9747066]",Apartment / Unit / Flat,1,1,0,"UNIT FOR LEASE - 2/9 CAMP STREET, DONALD",['* One bedroom unit in small complex.* All el...,...,POINT (142.9747066 -36.3700135),215031400.0,Buloke,NaN,NaN,False,Horsham,False,77.858022,243.561814
72,72,5/620 Princes Highway Mount Moriac VIC 3240,$220 pw,"[-38.2109634, 144.2355357]",Apartment / Unit / Flat,1,1,1,Fully furnished 1 bedroom unit on rural property,['* Please DO NOT attend this property without...,...,POINT (144.2355357 -38.2109634),203011036.0,Winchelsea,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,76,62 Bushlark Crs Williams Landing VIC 3027,$225 per week,"[-37.8553118, 144.7338452]",House,1,1,0,NEW GENERATION LIVING AT ITS FINEST! - Stunnin...,['New generation living at its finest! A selec...,...,POINT (144.7338452 -37.8553118),213051587.0,Truganina - South East,NaN,NaN,False,Melbourne,False,19.886952,19.886952
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11355,11355,30 Letchworth Avenue Brighton East VIC 3187,"$2,200.00","[-37.9289374, 145.0229494]",House,5,3,2,Hamptons Style Resort Living,"['Quietly concealed behind high secure fences,...",...,POINT (145.0229494 -37.9289374),208011170.0,Brighton East,NaN,NaN,False,Melbourne,False,11.578171,15.063367
11369,11369,3404/439 Collins Street Melbourne VIC 3004,$2400 Per Week,"[-37.8179178, 144.9601207]",Apartment / Unit / Flat,3,2,2,Executive home in the sky Fully Furnished,"[""Perfectly positioned within the uppermost le...",...,POINT (144.9601207 -37.8179178),206041505.0,Melbourne CBD - West,NaN,NaN,True,Melbourne,False,0.000000,1.314929
11397,11397,97 Punchbowl Road Flinders VIC 3929,$3200 Per Week,"[-38.4652721, 144.9797924]",House,4,3,2,Exclusively located in prestigious Punchbowl R...,"['Luxurious single-level family home, an enter...",...,POINT (144.9797924 -38.4652721),214021378.0,Flinders,NaN,NaN,False,Cranbourne,True,46.828516,72.768142
11400,11400,1706/499 St Kilda Road Melbourne VIC 3004,$3750 Per Week,"[-37.8433764, 144.9792605]",Apartment / Unit / Flat,5,5,4,Unrivalled penthouse living on Fawkner Park,['Establishing a benchmark for penthouse luxur...,...,POINT (144.9792605 -37.8433764),206041125.0,South Yarra - West,NaN,NaN,True,Melbourne,False,0.000000,1.314929
